# Reconstructing a long sequence with sliding windows

The joint SSH/SST model was trained on five-frame windows: two days at the
example tiles' 12-hour spacing. Here we reconstruct **20 frames (9.5 days)**
with the same trained model, using Rozet and Louppe's (2023) sliding-window
construction.

We compare two choices:

- **Sliding windows (`window_step=1`):** one diffusion process over all 20
  frames, using 16 overlapping five-frame windows at each step. Interior
  frames retain the prediction from the window centred on them; endpoints
  use the first or last window.
- **Independent blocks (`window_step=5`):** four non-overlapping five-frame
  reconstructions, with no exchange of information between blocks.

The main questions are whether sliding windows reduce boundary jumps,
whether they improve reconstruction errors, and how much more they cost.
These are separate questions; smoother transitions need not imply lower error.

**Requirements:** the [released models and example data](../README.md#download-models-and-example-data)
and a GPU. Allow roughly 15 minutes on an L40S for the illustrated setup;
runtime and memory use depend on your hardware and settings. The
[sliding-window guide](../docs/08_windowed_sda.md) explains the method.

### Why not generate one block after another?

An autoregressive rollout generates each new frame or block from earlier
states. If it freezes those earlier states, later observations cannot
revise them. That is restrictive for retrospective data assimilation: a
SWOT pass tomorrow may help reconstruct an eddy today.

Sliding-window SDA updates the whole sequence repeatedly, allowing
observations to influence earlier and later frames. An autoregressive model
can also support smoothing if it includes later observations or revisits
past states. **This notebook does not test that alternative**; its baseline
is independent blocks. See the guide's
[autoregression discussion](../docs/08_windowed_sda.md#why-not-generate-the-sequence-autoregressively).


In [ ]:
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

import obottle_starter as obs
from obottle_starter.inference.metrics import rmse
from obottle_starter.inference.osse import load_window
from obottle_starter.inference.sda.windows import window_layout

BUNDLE = Path("..")  # contains checkpoints/ and osse_tiles/
CKPT = BUNDLE / "checkpoints" / "joint_ssh_sst_T5.pt"
TILE = BUNDLE / "osse_tiles" / "MidAtlantic_35N_50W.nc"
device = "cuda" if torch.cuda.is_available() else "cpu"

net, cfg = obs.load_checkpoint(CKPT, device)
names = obs.channel_names(cfg)
T = cfg.time_length  # 5
L = 20  # frames to assimilate: (L - T) must be a multiple of the step
win = load_window(TILE, names, L)
print(f"{L} frames, {win.times[0]} -> {win.times[-1]}; the model sees {T} at a time")

## Which prediction is used for each frame?

`window_layout(L, T, step)` returns window start indices and identifies the
window and local position supplying each output frame. In the plot, each
row is a window: grey cells show the frames it reads, and dark cells show
the predictions it contributes to the assembled sequence.

Notice that overlapping windows share inputs, but their output predictions
are selected, not averaged.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for ax, step in zip(axes, [1, T]):
    starts, source, local = window_layout(L, T, step)
    grid = np.zeros((len(starts), L))
    for w, a in enumerate(starts):
        grid[w, a : a + T] = 1
    grid[source, np.arange(L)] = 2
    ax.imshow(grid, cmap="Greys", vmin=0, vmax=2.4, aspect="auto")
    ax.set_xlabel("frame")
    ax.set_ylabel("window")
    label = "sliding windows" if step == 1 else "independent blocks"
    ax.set_title(f"step = {step}: {len(starts)} windows ({label})")
plt.tight_layout()

## Reconstruct the same observations with both methods

Both runs use the same synthetic observation values, instrument noise, and
settings. Observation locations follow the tile's SWOT, nadir-altimeter,
and ACSPO coverage. We use **Rozet guidance**, which requires one backward
pass through the windows per sampling step. MMPS is another option, with
additional derivative evaluations for its linear solve.

**Choosing `γ`.** Rozet guidance uses variance `σ_y² + γ σ_t²`. Smaller
`γ` generally strengthens the correction. The current default is 0.1.
The recorded test with 0.01 diverged under this DDIM setup, even for a
single five-frame window, with SST errors of order 10⁸ °C.

Rozet and Louppe used 0.01 with Langevin corrector steps, which this sampler
does not implement. That difference motivates a comparison, but does not
by itself establish the cause of divergence. See
[the sampler discussion](../docs/04_inference_sda.md#langevin-corrector-steps-not-implemented-here).

**Memory:** guidance retains network activations for differentiation through
all windows. Reduce `n_batch` first if the run exceeds GPU memory. Limiting
windows per network call does not remove the need to retain those activations.


In [ ]:
common = dict(
    times=win.times,
    center_lon=win.center_lon,
    truth=win.truth,
    masks=win.masks,
    sigma_y={"SSH": 0.005, "SST": 0.1},
    guidance="rozet",
    n_members=4,
    n_batch=1,  # one member at a time: 16 windows take about 26 GB
    steps=64,
    seed=0,
    verbose=False,
)
posts = {}
for step in [T, 1]:
    t0 = time.time()
    posts[step] = obs.assimilate(net, cfg, window_step=step, **common)
    print(f"step {step}: {posts[step].attrs['n_windows']} windows, "
          f"{(time.time() - t0) / 60:.1f} min")
blocks, sliding = posts[T], posts[1]

## Reconstruction error at each frame

We compare the **ensemble-mean RMSE** against the simulated truth. Dashed
lines mark the boundaries between independent blocks. Look for changes
near those boundaries, but also inspect observation coverage: a period
with few observations can have large errors under either method.


In [ ]:
boundaries = np.arange(T, L, T) - 0.5
fig, axes = plt.subplots(1, len(names), figsize=(6 * len(names), 4), squeeze=False)
for ax, c in zip(axes[0], names):
    truth = blocks.truth.sel(channel=c).values
    for post, label in [(blocks, "independent blocks"), (sliding, "sliding windows")]:
        mean = post.samples.sel(channel=c).mean("sample").values
        ax.plot(rmse(mean, truth), "o-", label=label)
    for b in boundaries:
        ax.axvline(b, color="k", lw=0.5, ls="--")
    ax.set_xlabel("frame (12 h apart)")
    ax.set_ylabel(f"ensemble-mean RMSE ({'m' if c == 'SSH' else 'degC'})")
    ax.set_title(c)
    ax.legend()
plt.tight_layout()

## Temporal changes and block boundaries

For each member, we calculate the RMS change between consecutive frames,
then average over members and divide by the corresponding change in the
truth.

A ratio near 1 means the changes have similar RMS magnitude; it does not
establish that the fields or their dynamics are correct. Large spikes at
block boundaries indicate possible discontinuities. Elevated ratios away
from boundaries suggest a broader problem with temporal variability.


In [ ]:
def frame_jumps(field):
    # RMS change between consecutive frames: (..., L, H, W) -> (..., L - 1)
    return np.sqrt(np.mean(np.diff(field, axis=-3) ** 2, axis=(-2, -1)))


fig, axes = plt.subplots(1, len(names), figsize=(6 * len(names), 4), squeeze=False)
for ax, c in zip(axes[0], names):
    truth_jump = frame_jumps(blocks.truth.sel(channel=c).values)
    for post, label in [(blocks, "independent blocks"), (sliding, "sliding windows")]:
        member_jump = frame_jumps(post.samples.sel(channel=c).values).mean(0)
        ax.plot(np.arange(L - 1) + 0.5, member_jump / truth_jump, "o-", label=label)
    for b in boundaries:
        ax.axvline(b, color="k", lw=0.5, ls="--")
    ax.axhline(1, color="grey", lw=0.5)
    ax.set_xlabel("between frames")
    ax.set_ylabel("member change / truth change")
    ax.set_title(c)
    ax.legend()
plt.tight_layout()

## Inspect a time–space section

These panels show SSH along the middle grid row: horizontal position runs
across each panel and physical time runs downward. Compare the truth with
one member from each method, using the same colour scale.

Look for abrupt transitions at block boundaries and for differences in
structure between them. This visual check complements the error and
increment statistics; it cannot establish accuracy on its own.


In [ ]:
row = cfg.H // 2
panels = [
    (blocks.truth.sel(channel="SSH")[:, row], "truth"),
    (blocks.samples.sel(channel="SSH")[0, :, row], "independent blocks, member 0"),
    (sliding.samples.sel(channel="SSH")[0, :, row], "sliding windows, member 0"),
]
lim = float(np.abs(panels[0][0]).max())
fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharey=True)
for ax, (field, title) in zip(axes, panels):
    im = ax.imshow(field, cmap="RdBu_r", vmin=-lim, vmax=lim, aspect="auto")
    for b in boundaries:
        ax.axhline(b, color="k", lw=0.5, ls="--")
    ax.set_title(title)
    ax.set_xlabel("column (4 km)")
axes[0].set_ylabel("frame (12 h apart)")
plt.colorbar(im, ax=axes, shrink=0.8, label="SSH (m)")

## Does the longer sequence help the central frames?

We now reconstruct frames 8–12 as a single five-frame window and compare
with those same frames from the long reconstruction. Both runs receive
exactly the same observed values on these frames; the long reconstruction
also has observations from neighbouring days.

Those additional observations may help, but improved accuracy is not
guaranteed: the long sequence uses a window approximation, and a four-member
ensemble gives a noisy estimate of the posterior mean. This comparison is
not a numerical-equivalence test, because the inference problems differ.


In [ ]:
mid = slice(8, 8 + T)
# The same noisy observations the long run used, not a new noise draw.
y_mid = {c: sliding.observations.sel(channel=c).values[mid] for c in names}
single = obs.assimilate(
    net,
    cfg,
    **{**common, "times": win.times[mid], "n_batch": common["n_members"],
       "truth": {c: v[mid] for c, v in win.truth.items()},
       "masks": {c: v[mid] for c, v in win.masks.items()}, "y": y_mid},
)
for c in names:
    truth = single.truth.sel(channel=c).values
    one = rmse(single.samples.sel(channel=c).mean("sample").values, truth).mean()
    long = rmse(sliding.samples.sel(channel=c).mean("sample").values[mid], truth).mean()
    print(f"{c}: frames 8-12, RMSE one window {one:.4g}, sliding windows {long:.4g}")

## Interpreting the recorded example

The observations below describe the recorded MidAtlantic run with four
members and 64 steps. They are an illustration from one case, not a general
ranking of the methods. Check your outputs before expecting the same values.

- **Coverage helps explain error changes.** SSH swaths cover frames 2–6
  (13–24% of cells); frames 7–17 are almost unobserved. SST coverage falls
  to 3% at frames 5–6 and to zero at frame 14. Inspect `win.masks` alongside
  the error curves.
- **Sliding windows reduce block-boundary jumps.** At independent-block
  boundaries, SSH members change 5–7 times as much as the truth. The
  sliding-window result lacks those boundary spikes in this example.
  SST jumps in both methods near sharp changes in coverage.
- **Temporal variability remains too large.** Away from boundaries, SSH
  members still change 2–3 times as much as the truth. This occurs with both
  methods, so it is not explained by block boundaries alone. Possible
  contributions from the prior, guidance, and sampling settings need
  separate tests.
- **Accuracy is mixed.** Sliding windows improve SSH errors near observed
  frames 3–6 but worsen them at frames 7–9. On frames 8–12, SSH RMSE is
  0.061 m for sliding windows versus 0.070 m for the single-window run;
  SST RMSE is 0.123 °C versus 0.116 °C.
- **More windows cost more.** The recorded runs took 10.5 minutes for
  16 sliding windows and 2.4 minutes for four independent blocks.

The clearest improvement in this example is reduced boundary discontinuity.
The remaining variability and mixed error results are reasons to evaluate
more cases before drawing conclusions about reconstruction quality.

## Experiments to try next

- **An intermediate spacing:** `window_step=3` uses six windows for this
  20-frame example. Compare its cost, boundary behaviour, and error with
  steps 1 and 5.
- **MMPS:** set `guidance="mmps"` and retain the default `gmres` solver.
  The assembled Jacobian need not be symmetric; account for the extra cost.
- **The 11-frame SSH model:** try all 24 tile frames (`L=24`, 14 windows
  at step 1). It provides more temporal context, but also differs in variables
  and training, so this does not isolate window length.
- **Other locations and periods:** vary `t_start` and `origin` in
  `load_window`, ensuring that the requested sequence fits inside the tile.
- **Langevin corrector steps:** these require implementation changes; the
  [guide](../docs/04_inference_sda.md#langevin-corrector-steps-not-implemented-here)
  describes where to start. Compare their effect jointly with `γ` and
  sampling-step count.

The command-line equivalent uses `scripts/run_sda.py` with
`sliding.frames=20 guidance.method=rozet sampler.n_batch=1` in addition to
the checkpoint, tile, and output paths.
